<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thiết bị tính toán đang dùng: cpu


In [7]:
# ===== CELL 2: NẠP DỮ LIỆU 5 ĐẶC TRƯNG LÊN GPU =====
import numpy as np

X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy')), dtype=torch.float32, device=device)
S_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Steady_train.npy')), dtype=torch.float32, device=device)

X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy')), dtype=torch.float32, device=device)

X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy')), dtype=torch.float32, device=device)

print(f'Đã nạp lên {device}! Train: {X_train_t.shape} | Mẫu xác lập có tải: {int(S_train_t.sum())}')

Đã nạp lên cpu! Train: torch.Size([89974, 5]) | Mẫu xác lập có tải: 40857


In [8]:
# ===== CELL 3: KIẾN TRÚC PINN 5 ĐẦU VÀO & PHYSICS LOSS XÁC LẬP CHUẨN =====
import torch.nn as nn
import joblib

X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32, device=device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32, device=device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32, device=device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32, device=device)

class PMSM_PINN(nn.Module):
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, steady_mask, pole_pairs=4, psi_m=0.527139):
    mask = steady_mask > 0.5
    if mask.sum() == 0:
        return torch.tensor(0.0, device=inputs_scaled.device)

    in_s = inputs_scaled[mask]
    out_s = outputs_scaled[mask]

    inputs_real = in_s * x_scale + x_mean
    outputs_real = out_s * y_scale + y_mean

    ud = inputs_real[:, 0]
    uq = inputs_real[:, 1]
    Id = inputs_real[:, 2]
    Iq = inputs_real[:, 3]
    nm = inputs_real[:, 4]

    Rs_est = outputs_real[:, 0]
    Ld_est = outputs_real[:, 1]
    Lq_est = outputs_real[:, 2]

    omega_m = nm * (2.0 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    ud_calc = Rs_est * Id - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + omega_e * (Ld_est * Id + psi_m)

    # Chuẩn hóa sai số điện áp theo biên độ tải để Rs ở tốc độ thấp không bị át bởi uq ở tốc độ cao
    loss_ud = torch.mean(((ud - ud_calc) / 10.0) ** 2)
    loss_uq = torch.mean(((uq - uq_calc) / (torch.abs(uq) + 10.0)) ** 2)

    return loss_ud + loss_uq

model_pinn = PMSM_PINN(input_dim=5, output_dim=3, hidden_dim=128).to(device)
print("Đã khởi tạo mô hình PINN 5 đầu vào trên:", device)

Đã khởi tạo mô hình PINN 5 đầu vào trên: cpu


In [9]:
# ===== CELL 4: HUẤN LUYỆN PINN VỚI TRỌNG SỐ ƯU TIÊN CHO RS =====
import torch.optim as optim
import shutil
import time

# Trọng số ưu tiên học Rs (trọng số 3.0) so với Ld, Lq (1.0)
param_weights = torch.tensor([3.0, 1.0, 1.0], device=device)

def weighted_mse_loss(preds, targets):
    return torch.mean(param_weights * (preds - targets) ** 2)

optimizer = optim.AdamW(model_pinn.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

lambda_p = 0.05
epochs = 120
batch_size = 512
n_train = len(X_train_t)
best_val_loss = float('inf')

LOCAL_CKPT = '/content/best_pinn_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN PINN ===")
start_time = time.time()

for epoch in range(1, epochs + 1):
    model_pinn.train()
    perm = torch.randperm(n_train, device=device)
    train_data_loss = 0.0
    train_phys_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        inputs = X_train_t[idx]
        targets = Y_train_t[idx]
        steady = S_train_t[idx]

        optimizer.zero_grad()
        outputs = model_pinn(inputs)

        loss_d = weighted_mse_loss(outputs, targets)
        loss_p = compute_physics_loss(inputs, outputs, steady)
        loss_total = loss_d + lambda_p * loss_p

        loss_total.backward()
        optimizer.step()

        train_data_loss += loss_d.item() * len(idx)
        train_phys_loss += loss_p.item() * len(idx)

    train_data_loss /= n_train
    train_phys_loss /= n_train

    model_pinn.eval()
    with torch.no_grad():
        val_outputs = model_pinn(X_val_t)
        val_loss = weighted_mse_loss(val_outputs, Y_val_t).item()

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    if epoch % 15 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch:3d}/{epochs}] | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.6f} | Val Loss: {val_loss:.6f} | Thời gian: {elapsed:.1f} giây")

shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Hoàn thành trong {time.time() - start_time:.1f} giây! Đã lưu vào: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN PINN ===
Epoch [  1/120] | Data Loss: 0.936139 | Phys Loss: 0.255904 | Val Loss: 1.521812 | Thời gian: 1.1 giây
Epoch [ 15/120] | Data Loss: 0.631046 | Phys Loss: 0.254135 | Val Loss: 2.017981 | Thời gian: 13.6 giây
Epoch [ 30/120] | Data Loss: 0.552665 | Phys Loss: 0.254282 | Val Loss: 2.051031 | Thời gian: 26.7 giây
Epoch [ 45/120] | Data Loss: 0.524817 | Phys Loss: 0.254175 | Val Loss: 1.976871 | Thời gian: 39.8 giây
Epoch [ 60/120] | Data Loss: 0.514125 | Phys Loss: 0.254147 | Val Loss: 2.020638 | Thời gian: 53.8 giây
Epoch [ 75/120] | Data Loss: 0.510361 | Phys Loss: 0.254175 | Val Loss: 2.065712 | Thời gian: 67.2 giây
Epoch [ 90/120] | Data Loss: 0.508729 | Phys Loss: 0.254034 | Val Loss: 2.039839 | Thời gian: 80.3 giây
Epoch [105/120] | Data Loss: 0.508169 | Phys Loss: 0.254160 | Val Loss: 2.038342 | Thời gian: 93.9 giây
Epoch [120/120] | Data Loss: 0.507941 | Phys Loss: 0.254261 | Val Loss: 2.034686 | Thời gian: 107.7 giây

--> Hoàn thành trong 107.7 giây

In [10]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số tối ưu
model_pinn.eval()                                          # Bật chế độ dự đoán

with torch.no_grad():  # Tắt tính toán gradient khi kiểm tra tập Test
    # Dự đoán toàn bộ tập Test trực tiếp trên GPU và chuyển kết quả về CPU dạng mảng NumPy
    all_preds = model_pinn(X_test_t).cpu().numpy()
    all_targets = Y_test_t.cpu().numpy()

# Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực (Ohm cho Rs, Henry cho Ld và Lq)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 6.32%
Sai số Ld trung bình (PINN): 0.17%
Sai số Lq trung bình (PINN): 0.19%
